# 06 · Event catalogue and clustering

From triggers to a catalogue, and three separate meanings of "clustering":

1. **In time:** are events bunched more than a random (Poisson) process? Inter-event CV, Fano factor, burst sweep.
2. **Across channels:** do two sensors see the same events? Coincidence compared with a time-shifted chance level.
3. **By waveform:** are some events copies of each other (a rattling fitting, a dripping tap)? Cross-correlation
   clustering, plus per-event features (peak frequency, decay time, Q, energy) and feature-space clustering.

Read `docs/STUDENT_GUIDE.md` §7 and §12 first. Triggers are not bubbles, and correlation clusters of *bubbles*
are expected (similar-frequency ringdowns look alike).

In [ ]:
# ==== YOUR SETTINGS ====
CONFIG = "../config/demo.yaml"
TAKE = "DEMO_002"
CH = 3                       # channel for the waveform catalogue
BAND = "audio"               # detection band
WINDOW = (50.0, 175.0)       # [s] part of the take to analyse (e.g. after the acid)
MAX_EVENTS = 600             # waveforms to read (random subset if more)
PRE_S, POST_S = 0.002, 0.010 # snippet: 2 ms before to 10 ms after the trigger
RHO_MIN = 0.8                # correlation threshold for waveform clusters
N_FEATURE_CLUSTERS = 3

In [ ]:
# ---- standard setup (same in every notebook) ----
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import zoom_acoustics as za
from zoom_acoustics import plots as zp, dsp

cfg = za.load_config(CONFIG)
print("config   :", cfg.config_file)
print("data_dir :", cfg.data_dir)
print("cache_dir:", cfg.cache_dir)
print("figures  :", cfg.figures_dir)

In [ ]:
f = za.load_features(cfg, TAKE)
take = za.find_take(za.discover_takes(cfg.data_dir, recursive=cfg.recursive, pattern=cfg.file_pattern), TAKE)
from zoom_acoustics import catalogue as K
ev = f.events
ev = ev[(ev.band == BAND) & (ev.t_s >= WINDOW[0]) & (ev.t_s < WINDOW[1])]
ev = ev[~za.masks.mask_array(ev.t_s.to_numpy(), f.mask_windows)]
ev.groupby("channel").size().rename("triggers in window")

## 1. Clustering in time

CV = 1 and Fano = 1 for a Poisson process. Gaps that straddle a masked window are removed (they are holes
cut by the mask, not intervals). A burst count is meaningful only if it has a **plateau** over a range of gap values.

In [ ]:
rows = []
for c in f.channels:
    t = ev[ev.channel == c].t_s.to_numpy()
    st = K.interevent_stats(t, windows=f.mask_windows, dead_s=f.meta["detector"]["dead_s"], t_range=WINDOW)
    rows.append(dict(channel=c, label=f.label(c), **{k: v for k, v in st.items()}))
pd.DataFrame(rows).round(3)

In [ ]:
t = ev[ev.channel == CH].t_s.to_numpy()
fig = zp.plot_interevent(t, f.mask_windows, f.meta["detector"]["dead_s"], title=f"{f.label(CH)}")
zp.save(fig, cfg, f"{TAKE}_interevent_ch{CH}")
K.burst_sweep(t, windows=f.mask_windows)

## 2. Coincidence between channels

For each pair: the fraction of A's events with a B event within ±3 ms, compared with the same after shifting B by
several seconds (chance). Only the **excess** means anything. The lag is B minus A. A constant lag of a fraction of
a millisecond between two sensors a few cm apart is mostly instrument group delay, not travel time.

In [ ]:
rows = []
for a in f.channels:
    for b in f.channels:
        if a < b:
            r = K.coincidence(ev[ev.channel == a].t_s, ev[ev.channel == b].t_s, win_s=3e-3)
            rows.append(dict(A=a, B=b, **{k: v for k, v in r.items() if k != "lags_s"}))
pd.DataFrame(rows).round(3)

## 3. Waveforms, features and waveform clusters

Snippets are read from the WAV files (not the cache). Features per event: peak amplitude, energy, rise and decay
time, peak and centroid frequency, bandwidth, Q = π f τ, and the level of the second spectral peak.

In [ ]:
W, tt, sr = K.extract_waveforms(take, ev[ev.channel == CH].t_s, CH, PRE_S, POST_S, max_events=MAX_EVENTS)
feat_df = K.waveform_features(W, sr, PRE_S)
feat_df.insert(0, "t_s", tt)
feat_df.describe().round(3)

In [ ]:
C = K.xcorr_matrix(W)
lab, clusters, med = K.cluster_waveforms(C, rho_min=RHO_MIN, features=feat_df, times=tt)
print(f"median pairwise correlation of all events: {med:.2f}")
clusters.head(10).round(2)

**How to read this table.** Clusters are ranked by internal similarity (`median_rho`). A repeating mechanical
source stands out only if it is clearly *more* similar than the rest. Check `f_peak_hz`: a cluster on an apparatus
line (5.6 kHz in the Sep-2026 rig) is the line. `gap_cv` well below 1 means regular timing. Then **look at the
gallery.**

In [ ]:
top = clusters.cluster.head(5).tolist()
lab_show = np.where(np.isin(lab, top), lab, 0)
fig = zp.plot_waveform_gallery(W[lab_show > 0], sr, lab_show[lab_show > 0], pre_s=PRE_S,
                               title=f"{TAKE} {f.label(CH)}: five most self-similar clusters")
zp.save(fig, cfg, f"{TAKE}_gallery_ch{CH}")
fig = zp.plot_corr_matrix(C, lab)

### Feature-space clustering (descriptive)

Ward clustering of standardised log-features. **k is your choice**, so re-run with other k and other feature
sets, and only trust groups that persist.

In [ ]:
flab, scores, evr = K.feature_clusters(feat_df, n_clusters=N_FEATURE_CLUSTERS)
fig = zp.plot_feature_space(feat_df, flab, scores, evr)
zp.save(fig, cfg, f"{TAKE}_features_ch{CH}")
feat_df.assign(cluster=flab).groupby("cluster")[["f_peak_hz", "decay_ms", "q_est", "energy"]].median().round(4)

In [ ]:
out = feat_df.assign(xcorr_cluster=lab, feature_cluster=flab, channel=CH, take=TAKE)
out.to_csv(cfg.figures_path / f"{TAKE}_catalogue_ch{CH}.csv", index=False)
print("wrote", cfg.figures_path / f"{TAKE}_catalogue_ch{CH}.csv", len(out), "events")